Tonny Talukder
0432220005101050

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import pandas as pd

# Define the path to the CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/CSE-Data-Mining-Warehouse-Lab-426/Final Lab Test/chronic_kidney_disease.csv'

# Load the dataset, skipping bad lines and specifying the separator
df = pd.read_csv(csv_file_path, sep=',', on_bad_lines='skip')

# Display the first 5 rows of the DataFrame to verify loading
print("Dataset loaded successfully. Here are the first 5 rows:")
display(df.head())

# Display basic information about the dataset
print("\nDataset Info:")
df.info()

Dataset loaded successfully. Here are the first 5 rows:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     388 non-null    float64
 1   bp      385 non-null    float64
 2   sg      350 non-null    float64
 3   al      351 non-null    float64
 4   su      348 non-null    float64
 5   rbc     247 non-null    object 
 6   pc      332 non-null    object 
 7   pcc     393 non-null    object 
 8   ba      393 non-null    object 
 9   bgr     354 non-null    float64
 10  bu      378 non-null    float64
 11  sc      380 non-null    float64
 12  sod     312 non-null    float64
 13  pot     311 non-null    float64
 14  hemo    345 non-null    float64
 15  pcv     327 non-null    float64
 16  wbcc    292 non-null    float64
 17  rbcc    267 non-null    float64
 18  htn     395 non-null    object 
 19  dm      395 non-null    object 
 20  cad     395 non-null    object 
 21  appet   396 non-null    

In [6]:
# Step 1: Detect and remove outliers using only the integer columns (ignore binary and categorical columns).

# Identify numerical and categorical columns
# From df.info(), float64 columns are numerical, and object columns are categorical.
numerical_cols = df.select_dtypes(include=['float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns identified for outlier detection:", numerical_cols)
print("Categorical columns (ignored for outlier detection):")
for col in categorical_cols:
    print(f"- {col}: Unique values: {df[col].nunique()}")

# Impute missing values in numerical columns with the median
# This is important before applying outlier detection methods.
for col in numerical_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Missing values in '{col}' imputed with median: {median_val}")

print("\nMissing values after imputation:")
print(df[numerical_cols].isnull().sum())


Numerical columns identified for outlier detection: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
Categorical columns (ignored for outlier detection):
- rbc: Unique values: 2
- pc: Unique values: 2
- pcc: Unique values: 2
- ba: Unique values: 2
- htn: Unique values: 2
- dm: Unique values: 2
- cad: Unique values: 2
- appet: Unique values: 2
- pe: Unique values: 2
- ane: Unique values: 2
- class: Unique values: 2
Missing values in 'age' imputed with median: 54.5
Missing values in 'bp' imputed with median: 80.0
Missing values in 'sg' imputed with median: 1.02
Missing values in 'al' imputed with median: 0.0
Missing values in 'su' imputed with median: 0.0
Missing values in 'bgr' imputed with median: 121.0
Missing values in 'bu' imputed with median: 42.0
Missing values in 'sc' imputed with median: 1.3
Missing values in 'sod' imputed with median: 138.0
Missing values in 'pot' imputed with median: 4.4
Missing values in 'hemo' imputed with media

/tmp/ipykernel_3076/3140689531.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)


In [7]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import numpy as np

# Make a copy of the DataFrame for outlier detection to avoid modifying the original during detection
df_outliers = df.copy()

# --- i. IQR based outlier detection ---
print("\n--- Performing IQR based outlier detection ---")
outlier_indices_iqr = set()

for col in numerical_cols:
    Q1 = df_outliers[col].quantile(0.25)
    Q3 = df_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers for the current column
    col_outliers = df_outliers[(df_outliers[col] < lower_bound) | (df_outliers[col] > upper_bound)].index
    outlier_indices_iqr.update(col_outliers)
    if len(col_outliers) > 0:
        print(f"Column '{col}': {len(col_outliers)} IQR outliers detected.")

print(f"Total unique IQR outliers detected across all numerical columns: {len(outlier_indices_iqr)}")

# --- ii. One-Class SVM ---
print("\n--- Performing One-Class SVM based outlier detection ---")

# Scale numerical data for One-Class SVM
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_outliers[numerical_cols])

# Initialize One-Class SVM. Nu is the proportion of outliers expected (0.01 to 0.1 is common).
# A lower nu makes the model more sensitive to outliers.
# Let's choose nu=0.05, meaning we expect about 5% of the data to be outliers.
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.05)

# Fit the model and predict outliers (-1 for outliers, 1 for inliers)
ocsvm.fit(df_scaled)
predictions = ocsvm.predict(df_scaled)

outlier_indices_ocsvm = set(df_outliers.index[predictions == -1])

print(f"Total One-Class SVM outliers detected: {len(outlier_indices_ocsvm)}")

# --- Combine results and remove outliers ---
# Combine outlier indices from both methods
all_outlier_indices = list(outlier_indices_iqr.union(outlier_indices_ocsvm))

print(f"\nTotal unique outliers identified by either IQR or One-Class SVM: {len(all_outlier_indices)}")

# Remove identified outlier rows from the DataFrame
df_cleaned = df.drop(all_outlier_indices).reset_index(drop=True)

print(f"Original dataset shape: {df.shape}")
print(f"Dataset shape after outlier removal: {df_cleaned.shape}")

# Update the main DataFrame 'df' with the cleaned version
df = df_cleaned



--- Performing IQR based outlier detection ---
Column 'age': 10 IQR outliers detected.
Column 'bp': 36 IQR outliers detected.
Column 'sg': 7 IQR outliers detected.
Column 'su': 59 IQR outliers detected.
Column 'bgr': 52 IQR outliers detected.
Column 'bu': 38 IQR outliers detected.
Column 'sc': 51 IQR outliers detected.
Column 'sod': 18 IQR outliers detected.
Column 'pot': 14 IQR outliers detected.
Column 'hemo': 2 IQR outliers detected.
Column 'pcv': 6 IQR outliers detected.
Column 'wbcc': 17 IQR outliers detected.
Column 'rbcc': 74 IQR outliers detected.
Total unique IQR outliers detected across all numerical columns: 191

--- Performing One-Class SVM based outlier detection ---
Total One-Class SVM outliers detected: 29

Total unique outliers identified by either IQR or One-Class SVM: 191
Original dataset shape: (397, 25)
Dataset shape after outlier removal: (206, 25)


In [10]:
# Step 2: Remove all numerical columns from the dataset
# The 'numerical_cols' list was already identified in the previous step.

print(f"Dataset shape before removing numerical columns: {df.shape}")
df_prepared = df.drop(columns=numerical_cols)
print(f"Dataset shape after removing numerical columns: {df_prepared.shape}")

# Step 3: Convert all categorical columns into binary columns using one-hot encoding.

# Identify categorical columns that are still in df_prepared
# (These are the original 'categorical_cols' from df, minus any that might have been dropped, though none were in this case)
categorical_cols_for_ohe = df_prepared.select_dtypes(include=['object']).columns.tolist()

print(f"\nCategorical columns identified for one-hot encoding: {categorical_cols_for_ohe}")

# Perform one-hot encoding, and then explicitly convert to boolean type for mlxtend compatibility
df_encoded = pd.get_dummies(df_prepared, columns=categorical_cols_for_ohe, prefix=categorical_cols_for_ohe, dtype=int).astype(bool)

print(f"\nDataset shape after one-hot encoding: {df_encoded.shape}")
print("First 5 rows of the encoded dataset:")
display(df_encoded.head())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Dataset shape before removing numerical columns: (206, 25)
Dataset shape after removing numerical columns: (206, 11)

Categorical columns identified for one-hot encoding: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']

Dataset shape after one-hot encoding: (206, 22)
First 5 rows of the encoded dataset:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,rbc_abnormal,rbc_normal,pc_abnormal,pc_normal,pcc_notpresent,pcc_present,ba_notpresent,ba_present,htn_no,htn_yes,...,cad_no,cad_yes,appet_good,appet_poor,pe_no,pe_yes,ane_no,ane_yes,class_ckd,class_notckd
0,False,False,False,True,True,False,True,False,False,True,...,True,False,True,False,True,False,True,False,True,False
1,False,True,False,True,True,False,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False
2,False,False,False,False,True,False,True,False,False,True,...,True,False,True,False,False,True,True,False,True,False
3,False,False,False,False,True,False,True,False,False,True,...,False,True,False,True,False,True,True,False,True,False
4,False,False,False,True,True,False,True,False,True,False,...,True,False,True,False,True,False,True,False,True,False


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [11]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# Step 4: Apply the FP-Growth algorithm to find association rules from the final dataset.

# Find frequent itemsets using FP-Growth
# min_support is a crucial parameter. Choosing a value that balances performance and meaningful results.
# A lower support value will result in more frequent itemsets but will also take longer to compute.
# Given the reduced dataset size after outlier removal (206 rows), a min_support of 0.1 (10%) is a reasonable starting point.
print("\n--- Finding Frequent Itemsets using FP-Growth ---")
frequent_itemsets = fpgrowth(df_encoded, min_support=0.1, use_colnames=True)

print(f"Found {len(frequent_itemsets)} frequent itemsets.")
print("First 5 frequent itemsets:")
display(frequent_itemsets.head())

# Generate association rules from frequent itemsets
# min_confidence is a crucial parameter. A common starting value is 0.5 or 0.7.
# We will use a min_confidence of 0.7 for strong rules.
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print(f"\nFound {len(rules)} association rules with min_confidence = 0.7.")
print("First 5 association rules (unsorted):")
display(rules.head())

# Step 5: Report the top 10 association rules by sorting them based on confidence.
# If confidence score is same for two rules, sort by support.

print("\n--- Top 10 Association Rules (sorted by Confidence then Support) ---")
top_10_rules = rules.sort_values(by=['confidence', 'support'], ascending=[False, False]).head(10)
display(top_10_rules)

# Optional: Display more metrics for understanding the rules
# print("\nAll association rules with additional metrics:")
# display(rules.sort_values(by=['confidence', 'lift'], ascending=[False, False]))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


--- Finding Frequent Itemsets using FP-Growth ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Found 2640 frequent itemsets.
First 5 frequent itemsets:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


--- Top 10 Association Rules (sorted by Confidence then Support) ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
122,"(pcc_notpresent, ane_no, appet_good)",(ba_notpresent),0.815534,0.966019,0.815534,1.0,1.035176,1.0,0.027712,inf,0.184211,0.844221,1.0,0.922111
153,"(pcc_notpresent, cad_no, appet_good, ane_no)",(ba_notpresent),0.800971,0.966019,0.800971,1.0,1.035176,1.0,0.027217,inf,0.170732,0.829146,1.0,0.914573
350,"(pcc_notpresent, pe_no, appet_good)",(ba_notpresent),0.781553,0.966019,0.781553,1.0,1.035176,1.0,0.026558,inf,0.155556,0.809045,1.0,0.904523
381,"(pcc_notpresent, pe_no, cad_no, appet_good)",(ba_notpresent),0.766990,0.966019,0.766990,1.0,1.035176,1.0,0.026063,inf,0.145833,0.793970,1.0,0.896985
18463,"(ane_no, htn_no)",(cad_no),0.762136,0.970874,0.762136,1.0,1.030000,1.0,0.022198,inf,0.122449,0.785000,1.0,0.892500
484,"(pcc_notpresent, pe_no, ane_no, appet_good)",(ba_notpresent),0.757282,0.966019,0.757282,1.0,1.035176,1.0,0.025733,inf,0.140000,0.783920,1.0,0.891960
544,"(cad_no, ane_no, pcc_notpresent, appet_good, p...",(ba_notpresent),0.747573,0.966019,0.747573,1.0,1.035176,1.0,0.025403,inf,0.134615,0.773869,1.0,0.886935
17169,"(ane_no, pcc_notpresent, dm_no)",(cad_no),0.737864,0.970874,0.737864,1.0,1.030000,1.0,0.021491,inf,0.111111,0.760000,1.0,0.880000
17198,"(ba_notpresent, ane_no, pcc_notpresent, dm_no)",(cad_no),0.733010,0.970874,0.733010,1.0,1.030000,1.0,0.021350,inf,0.109091,0.755000,1.0,0.877500
18483,"(ane_no, dm_no, htn_no)",(cad_no),0.733010,0.970874,0.733010,1.0,1.030000,1.0,0.021350,inf,0.109091,0.755000,1.0,0.877500


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag